In [1]:
import os
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet101
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
import time

In [ ]:
# Set constants and paths
batch_size = 32
learning_rate = 0.001
num_epochs = 20
RANDOM_STATE = 1082
MODEL_NAME = "ResNet-CNN"


In [ ]:
import sys as _sys
from pathlib import Path as _Path

_root = _Path.cwd().resolve()
while not (_root / "eval_metrics.py").is_file() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in _sys.path:
    _sys.path.insert(0, str(_root))
from gps_ids import images

IMAGES_CONFIG = images.UAV_IMAGES
device = images.seed_everything()


In [ ]:
IMG = images.loaders(IMAGES_CONFIG, batch_size=batch_size)
dataloaders = IMG.dataloaders
classes = IMG.folder_classes


In [6]:
# Model
class CNN_Model(nn.Module):
    def __init__(self, num_classes=2, feature_dim=300):
        super(CNN_Model, self).__init__()
        self.resnet = resnet101(pretrained=True)
        # Change first conv layer for grayscale
        self.resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, feature_dim)
        self.fc1 = nn.Linear(feature_dim, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        features = self.resnet(x)
        x = F.relu(self.fc1(features))
        x = self.fc2(x)
        return x

In [7]:
# Instantiate model, loss, optimizer
model = CNN_Model(num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)

C:\Users\antho\miniforge3\envs\pytorch\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\antho\miniforge3\envs\pytorch\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [8]:
# Training loop
best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0
best_epoch = 0

start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    all_train_preds = []
    all_train_labels = []
    for inputs, labels in dataloaders['train']:
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        all_train_preds.append(preds.cpu().numpy())
        all_train_labels.append(labels.cpu().numpy())
    epoch_loss = running_loss / len(train_dataset)
    train_preds = np.concatenate(all_train_preds)
    train_labels = np.concatenate(all_train_labels)
    train_acc = accuracy_score(train_labels, train_preds)

    # Validation on held-out validation set
    model.eval()
    all_val_preds = []
    all_val_labels = []
    with torch.no_grad():
        for inputs, labels in dataloaders['val']:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_val_preds.append(preds.cpu().numpy())
            all_val_labels.append(labels.cpu().numpy())
    val_preds = np.concatenate(all_val_preds)
    val_labels = np.concatenate(all_val_labels)
    val_acc = accuracy_score(val_labels, val_preds)
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_loss:.4f} - Train Acc: {train_acc:.4f} - Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())

train_time = time.time() - start_time

model.load_state_dict(best_model_wts)

Epoch 1/20 - Train Loss: 0.4398 - Train Acc: 0.8448 - Val Acc: 0.8631
Epoch 2/20 - Train Loss: 0.3931 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 3/20 - Train Loss: 0.3688 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 4/20 - Train Loss: 0.3298 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 5/20 - Train Loss: 0.2718 - Train Acc: 0.8658 - Val Acc: 0.8631
Epoch 6/20 - Train Loss: 0.1949 - Train Acc: 0.9241 - Val Acc: 0.8921
Epoch 7/20 - Train Loss: 0.1448 - Train Acc: 0.9569 - Val Acc: 0.9530
Epoch 8/20 - Train Loss: 0.0884 - Train Acc: 0.9738 - Val Acc: 0.9488
Epoch 9/20 - Train Loss: 0.0790 - Train Acc: 0.9759 - Val Acc: 0.9834
Epoch 10/20 - Train Loss: 0.0590 - Train Acc: 0.9831 - Val Acc: 0.9834
Epoch 11/20 - Train Loss: 0.0606 - Train Acc: 0.9817 - Val Acc: 0.9862
Epoch 12/20 - Train Loss: 0.0459 - Train Acc: 0.9859 - Val Acc: 0.9931
Epoch 13/20 - Train Loss: 0.0417 - Train Acc: 0.9865 - Val Acc: 0.9931
Epoch 14/20 - Train Loss: 0.0362 - Train Acc: 0.9865 - Val Acc: 0.9917
Epoch 15/20 - T

<All keys matched successfully>

In [ ]:
# Final evaluation
model.eval()
all_preds = []
all_probs = []
all_labels = []
with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"Test F1 Score: {f1:.4f}")

In [ ]:
from eval_metrics import evaluate, summary, time_inference_torch
from gps_ids.results import append_validated

_timing = time_inference_torch(model, next(iter(dataloaders['val']))[0].to(device))

row = evaluate(
    all_labels,
    all_preds,
    model_name=MODEL_NAME,
    family="Image",
    y_proba=all_probs,
    train_time_sec=train_time,
    num_epochs=num_epochs,
    parameters=list(dict(model.named_parameters()).keys()),
    timing=_timing,
    **IMG.eval_kwargs(),
)
append_validated(row, IMG.results_csv)
print(summary(row))
print(f"Results appended to {IMG.results_csv}")


In [ ]:
# Save best model
model_save_path = str(images.checkpoint_path(IMAGES_CONFIG, "resnet"))
checkpoint = {
    'epoch': best_epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict()
}
torch.save(checkpoint, model_save_path)
print(f"Model saved to {model_save_path}")